# Resultados de regresión: log(1 + casos) en $t + 4$

Siete modelos × cuatro optimizadores = 28 combinaciones, con la misma validación anidada (criterio: −RMSE).

In [1]:
import os  # Rutas
import sys  # Configuración del intérprete
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count()))  # Evita un aviso de joblib en Windows
sys.path[:0] = [os.path.abspath(".."), os.path.abspath("../.."), os.path.abspath("../../entregable1")]  # e3, src y e1
import warnings  # Control de avisos
warnings.filterwarnings("ignore")  # Oculta avisos de versiones
import contextlib, io  # Silenciar un diagnóstico de joblib en Windows
from joblib.externals.loky import cpu_count  # Conteo de núcleos
with contextlib.redirect_stderr(io.StringIO()):  # joblib imprime un diagnóstico al no encontrar «wmic»
    cpu_count(only_physical_cores=True)  # Se calcula una vez (queda en caché)
import pickle  # Lectura de resultados
import numpy as np  # Cálculo numérico
import pandas as pd  # Tablas
import plotly.express as px  # Figuras rápidas
import plotly.graph_objects as go  # Figuras detalladas
from plotly.subplots import make_subplots  # Paneles
from IPython.display import Markdown, display  # Texto enriquecido
from e1.figuras import mostrar, AZUL, ROJO, VERDE, AMBAR, GRIS, MORADO  # Estilo de figuras (tema claro del informe)
from e3 import config as c3, datos as dd, modelos as mo, optimizadores as op, experimento as ex  # Código del entregable
c3.fijar_semilla()  # Semilla global (42)
pd.set_option("display.width", 170)  # Ancho de las tablas
pd.set_option("display.max_columns", 30)  # Columnas visibles
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")  # Cuatro decimales
COLORES = {"Grid Search": GRIS, "Random Search": AMBAR, "Bayesiana (Optuna TPE)": AZUL, "Genética (DEAP)": VERDE, "Successive Halving": MORADO}  # Color por optimizador
from e3 import evaluacion as ev  # Métricas y pruebas de residuos
tabla = pd.read_csv(c3.RESULTADOS / "tabla_maestra.csv")  # Tabla maestra
reg = tabla[tabla["tarea"] == "reg"]  # Regresión
externo = reg.groupby(["modelo", "optimizador"]).agg(**{"RMSE media": ("RMSE", "mean"), "RMSE desv.": ("RMSE", "std"), "MAE media": ("MAE", "mean"),
                                                      "MAE desv.": ("MAE", "std"), "R² media": ("R2", "mean"), "segundos": ("segundos_optimizacion", "sum")}).sort_values("RMSE media")
externo

RMSE media  RMSE desv.  MAE media  MAE desv.  R² media   segundos
modelo            optimizador                                                                              
XGBoost           Genética (DEAP)             0.4808      0.0790     0.3390     0.0709    0.7292   195.2334
                  Bayesiana (Optuna TPE)      0.4810      0.0785     0.3434     0.0665    0.7290   171.0359
                  Random Search               0.4811      0.0774     0.3426     0.0675    0.7287   205.8444
SVR               Genética (DEAP)             0.4823      0.0761     0.3436     0.0628    0.7271   214.5594
Random Forest     Bayesiana (Optuna TPE)      0.4824      0.0778     0.3444     0.0669    0.7272 1,486.7586
                  Genética (DEAP)             0.4830      0.0771     0.3444     0.0658    0.7263 1,022.4648
SVR               Bayesiana (Optuna TPE)      0.4831      0.0767     0.3486     0.0584    0.7263   221.4426
XGBoost           Grid Search                 0.4834      0.0781     0.3484     0.0663    0.7262   215.8909
Random Forest     Grid Search                 0.4834      0.0791     0.3460     0.0672    0.7263   974.1620
SVR               Grid Search                 0.4836      0.0780     0.3429     0.0660    0.7259   162.1708
Random Forest     Random Search               0.4839      0.0787     0.3452     0.0672    0.7256 1,258.6151
SVR               Random Search               0.4853      0.0771     0.3460     0.0622    0.7238   208.9416
Ridge             Grid Search                 0.4870      0.0745     0.3399     0.0670    0.7213     7.8507
                  Bayesiana (Optuna TPE)      0.4870      0.0745     0.3399     0.0670    0.7213     8.6733
                  Random Search               0.4870      0.0744     0.3399     0.0669    0.7213     8.3755
                  Genética (DEAP)             0.4872      0.0742     0.3405     0.0660    0.7210     9.0401
Lasso             Random Search               0.4872      0.0745     0.3399     0.0663    0.7210    26.4001
                  Genética (DEAP)             0.4872      0.0745     0.3401     0.0662    0.7210    53.1358
                  Bayesiana (Optuna TPE)      0.4879      0.0755     0.3405     0.0669    0.7204    56.8271
                  Grid Search                 0.4881      0.0758     0.3408     0.0674    0.7203    25.0515
Árbol de decisión Genética (DEAP)             0.4943      0.0784     0.3526     0.0643    0.7133    39.6141
                  Bayesiana (Optuna TPE)      0.4970      0.0825     0.3548     0.0678    0.7108    46.9530
                  Random Search               0.4977      0.0802     0.3558     0.0655    0.7096    61.6090
                  Grid Search                 0.5021      0.0817     0.3544     0.0675    0.7045    64.1136
KNN               Grid Search                 0.5139      0.0895     0.3537     0.0754    0.6917    65.0938
                  Bayesiana (Optuna TPE)      0.5140      0.0896     0.3543     0.0758    0.6916    63.0133
                  Random Search               0.5145      0.0893     0.3541     0.0753    0.6910    65.4963
                  Genética (DEAP)             0.5150      0.0891     0.3540     0.0756    0.6903    63.0041

In [2]:
tr, va, te = dd.cargar()  # Periodos
finales = {}  # Predicciones de las 28 combinaciones
for f in sorted(ex.FINALES.glob("reg__*.pkl")):  # Reajustes finales
    r = pickle.load(open(f, "rb"))  # Predicciones
    finales[(r["modelo"], r["optimizador"])] = r  # Indexado
mejores = externo.reset_index().groupby("modelo").first().sort_values("RMSE media")  # Mejor optimizador de cada modelo (bucle externo)
sel = [(m, o) for m, o in mejores["optimizador"].items()]  # Las 7 seleccionadas
prueba = pd.DataFrame({f"{m} · {o}": {**ev.metricas_reg(te["y_reg"], finales[(m, o)]["p_te"]), **{f"{k} (validación 2022)": v for k, v in ev.metricas_reg(va["y_reg"], finales[(m, o)]["p_va"]).items()}}
                       for m, o in sel}).T  # Métricas en la prueba y en 2022
prueba["RMSE persistencia (prueba)"] = np.sqrt(np.mean((te["y_reg"] - te["lc_l0"]) ** 2))  # Línea base trivial
prueba.sort_values("RMSE")

,RMSE,MAE,R2,RMSE (validación 2022),MAE (validación 2022),R2 (validación 2022),RMSE persistencia (prueba)
XGBoost · Genética (DEAP),0.5734,0.4381,0.7873,0.4549,0.3155,0.7560,0.6585
SVR · Genética (DEAP),0.5771,0.4455,0.7846,0.4559,0.3234,0.7550,0.6585
Random Forest · Bayesiana (Optuna TPE),0.5778,0.4427,0.7840,0.4549,0.3198,0.7560,0.6585
Lasso · Random Search,0.5824,0.4408,0.7806,0.4603,0.3137,0.7502,0.6585
Ridge · Grid Search,0.5830,0.4412,0.7801,0.4604,0.3138,0.7501,0.6585
Árbol de decisión · Genética (DEAP),0.5967,0.4512,0.7697,0.4658,0.3288,0.7442,0.6585
KNN · Grid Search,0.6314,0.4720,0.7421,0.4850,0.3228,0.7227,0.6585


**Interpretación.** En el bucle externo XGBoost (RMSE 0,481 ± 0,079), SVR con núcleo aproximado y Random Forest quedan prácticamente empatados; Ridge y Lasso, a menos de 0,01; el árbol único y KNN, detrás. Las diferencias entre optimizadores son mínimas (milésimas). En la prueba 2023–2024 XGBoost logra RMSE 0,573, MAE 0,438 y $R^2$ 0,79, **13 % menos error que la persistencia** (0,659). Las métricas empeoran de 2022 (RMSE 0,455) a la prueba porque 2023–2024 tiene más casos y más variación.

## Ajuste en entrenamiento, validación y prueba

Para ver el ajuste en cada subconjunto se promedian, por semana, el valor observado y el predicho de los 376 municipios (arriba), y se muestra la serie de un municipio (abajo).

In [3]:
m, o = sel[0]  # Mejor regresor
r = finales[(m, o)]  # Sus predicciones
ent = tr.loc[r["idx_tr"]]  # Submuestra de entrenamiento del reajuste final
partes = [("entrenamiento", ent, r["p_tr"], AZUL), ("validación 2022", va, r["p_va"], VERDE), ("prueba 2023–2024", te, r["p_te"], ROJO)]  # Subconjuntos
fig = make_subplots(2, 1, subplot_titles=(f"Promedio semanal de los municipios — {m} ({o})", "Santiago de Cali"), vertical_spacing=0.12)  # Dos paneles
for nombre, df_, pred, col in partes:  # Cada subconjunto
    s = pd.DataFrame({"fecha": df_["week_start"].to_numpy(), "obs": df_["y_reg"].to_numpy(), "pred": pred}).groupby("fecha").mean()  # Semana
    fig.add_scatter(x=s.index, y=s["obs"], line=dict(color="#94a3b8", width=1), name="observado" if nombre == "entrenamiento" else None, showlegend=nombre == "entrenamiento", row=1, col=1)  # Observado
    fig.add_scatter(x=s.index, y=s["pred"], line=dict(color=col, width=2), name=f"predicho ({nombre})", row=1, col=1)  # Predicho
    cali = df_["COD_MUN_N"].to_numpy() == 76001  # Cali
    fig.add_scatter(x=df_["week_start"].to_numpy()[cali], y=df_["y_reg"].to_numpy()[cali], mode="markers", marker=dict(color="#94a3b8", size=4), showlegend=False, row=2, col=1)  # Observado
    fig.add_scatter(x=df_["week_start"].to_numpy()[cali], y=np.asarray(pred)[cali], mode="markers", marker=dict(color=col, size=4), showlegend=False, row=2, col=1)  # Predicho
fig.update_yaxes(title_text="log(1 + casos en t + 4)")  # Eje
mostrar(fig, 620)  # Muestra

**Interpretación.** El modelo sigue bien el nivel medio en los tres subconjuntos, incluida la epidemia de 2024, aunque **suaviza los picos** y llega con algo de retraso a los cambios bruscos: predice a 4 semanas con la información de la semana actual. En Cali (abajo) se ve lo mismo a escala municipal: buen seguimiento de la tendencia, subestimación de los máximos.

## Análisis de residuos

* **White** (homocedasticidad): regresión auxiliar de $e^2$ sobre el valor ajustado, la temperatura, sus cuadrados y su producto.
* **BDS** (independencia, Brock-Dechert-Scheinkman): sobre la serie semanal del residuo medio; detecta dependencia lineal y no lineal.
* **Ljung-Box**: autocorrelación serial de la misma serie (rezagos 1, 4 y 8) y proporción de municipios con dependencia.
* **Jarque-Bera**: normalidad (asimetría y curtosis).

In [4]:
res = pd.DataFrame({f"{m} · {o}": ev.pruebas_residuos(te, te["y_reg"].to_numpy(), finales[(m, o)]["p_te"]) for m, o in sel}).T  # Pruebas
res[["RMSE"]] = prueba[["RMSE"]]  # RMSE junto a las pruebas
res[["RMSE", "p White", "p BDS", "p Ljung-Box (1)", "p Ljung-Box (4)", "municipios con Ljung-Box p < 0,05", "p Jarque-Bera", "asimetría", "curtosis"]]

,RMSE,p White,p BDS,p Ljung-Box (1),p Ljung-Box (4),"municipios con Ljung-Box p < 0,05",p Jarque-Bera,asimetría,curtosis
XGBoost · Genética (DEAP),0.5734,0.0000,0.0000,0.0000,0.0000,0.8856,0.0000,0.3057,0.6506
SVR · Genética (DEAP),0.5771,0.0000,0.0000,0.0000,0.0000,0.8670,0.0000,0.3140,0.5320
Random Forest · Bayesiana (Optuna TPE),0.5778,0.0000,0.0000,0.0000,0.0000,0.8830,0.0000,0.3105,0.5912
Ridge · Grid Search,0.5830,0.0000,0.0000,0.0000,0.0000,0.8936,0.0000,0.2182,0.6516
Lasso · Random Search,0.5824,0.0000,0.0000,0.0000,0.0000,0.8936,0.0000,0.2181,0.6483
Árbol de decisión · Genética (DEAP),0.5967,0.0000,0.0000,0.0000,0.0000,0.8963,0.0000,0.3133,0.6824
KNN · Grid Search,0.6314,0.0000,0.0000,0.0000,0.0000,0.8910,0.0000,0.2956,0.5207


In [5]:
from statsmodels.tsa.stattools import acf  # Autocorrelación
fig = make_subplots(2, 2, subplot_titles=("Residuos frente al valor ajustado (muestra)", "Histograma de residuos", "ACF del residuo medio semanal", "ACF media dentro de los municipios"))  # Paneles
m, o = sel[0]  # Mejor regresor
f_te = finales[(m, o)]["p_te"].astype(float)  # Predicción
e = te["y_reg"].to_numpy() - f_te  # Residuos
i = np.random.default_rng(c3.SEMILLA).choice(len(e), 6000, replace=False)  # Muestra
fig.add_scattergl(x=f_te[i], y=e[i], mode="markers", marker=dict(size=3, color=AZUL, opacity=0.4), showlegend=False, row=1, col=1)  # Dispersión
h = np.histogram(e, 80)  # Histograma
fig.add_bar(x=(h[1][:-1] + h[1][1:]) / 2, y=h[0], marker_color=AZUL, showlegend=False, row=1, col=2)  # Barras
semanal = pd.Series(e, index=te["time_idx"].to_numpy()).groupby(level=0).mean()  # Serie semanal
a = acf(semanal, nlags=12)  # ACF
fig.add_bar(x=np.arange(1, 13), y=a[1:], marker_color=AMBAR, showlegend=False, row=2, col=1)  # ACF semanal
for cc in (1, 2):  # Bandas
    fig.add_hline(y=1.96 / np.sqrt(len(semanal)), line_dash="dot", line_color=GRIS, row=2, col=cc)
    fig.add_hline(y=-1.96 / np.sqrt(len(semanal)), line_dash="dot", line_color=GRIS, row=2, col=cc)
acf_mun = np.nanmean([acf(g.to_numpy(), nlags=12)[1:] for _, g in pd.Series(e, index=te["COD_MUN_N"].to_numpy()).groupby(level=0) if g.std() > 0], axis=0)  # ACF por municipio
fig.add_bar(x=np.arange(1, 13), y=acf_mun, marker_color=VERDE, showlegend=False, row=2, col=2)  # ACF media
fig.update_layout(title=f"Diagnóstico de residuos en la prueba — {m} ({o})")  # Título
mostrar(fig, 620)  # Muestra

**Interpretación.** Los supuestos de independencia y homocedasticidad **no se cumplen** en ningún modelo: White rechaza la homocedasticidad (la varianza del error crece con el nivel de casos, como anticipó el EDA), BDS rechaza la independencia y Ljung-Box encuentra autocorrelación en la serie semanal y en ≈ 89 % de los municipios. Los residuos son levemente asimétricos y con colas un poco más pesadas que la normal. Es la firma de información estructural sin capturar (rachas epidémicas y dinámica de vecindad). Para la predicción puntual no invalida el modelo, pero obliga a usar pruebas que toleran dependencia en la comparación (bootstrap estacionario, MCS) y sugiere modelos con rezagos espaciales o del propio error.